## 1. Initialize Project Environment
Import libraries for enrichment analysis and biological interpretation.

In [1]:
from __future__ import annotations

import logging
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Dict, List, Optional, Set

import numpy as np
import pandas as pd
from scipy import stats

# Import gseapy for real KEGG/GO enrichment via Enrichr
try:
    import gseapy as gp

    GSEAPY_AVAILABLE = True
    print(f"gseapy {gp.__version__} available - using Enrichr/KEGG/GO databases")
except ImportError:
    GSEAPY_AVAILABLE = False
    print("gseapy not available - using local Fisher's test")

logging.basicConfig(level=logging.INFO, format="[%(levelname)s] %(message)s")

print("pandas", pd.__version__)
print("numpy", np.__version__)

gseapy 1.1.11 available - using Enrichr/KEGG/GO databases
pandas 2.2.3
numpy 2.1.3


## 2. Define Configuration Parameters
Centralize enrichment analysis settings.

In [2]:
@dataclass
class EnrichmentConfig:
    modules_file: Path = Path("artifacts/task3_module_assignments.csv")
    hub_file: Path = Path("artifacts/task3_hub_genes.csv")
    export_dir: Path = Path("artifacts")
    target_gene: str = "TP53"  # Focus analysis on TP53's module
    pvalue_threshold: float = 0.05
    min_genes_for_enrichment: int = 5

    def describe(self) -> Dict[str, str]:
        info = asdict(self)
        for k in ["modules_file", "hub_file", "export_dir"]:
            info[k] = str(info[k])
        return info


CONFIG = EnrichmentConfig()
CONFIG.describe()

{'modules_file': 'artifacts/task3_module_assignments.csv',
 'hub_file': 'artifacts/task3_hub_genes.csv',
 'export_dir': 'artifacts',
 'target_gene': 'TP53',
 'pvalue_threshold': 0.05,
 'min_genes_for_enrichment': 5}

## 3. Load Module Data
Load detected modules and hub gene information.

In [3]:
# Load data
modules = pd.read_csv(CONFIG.modules_file)
hubs = pd.read_csv(CONFIG.hub_file)

logging.info(f"Loaded {len(modules)} genes with module assignments")
logging.info(f"Modules: {modules['Module'].unique().tolist()}")

# Check for hub column
if "IsHub" in hubs.columns:
    logging.info(f"Hub genes: {hubs['IsHub'].sum()}")
else:
    logging.info(f"Hub genes file contains {len(hubs)} entries")

# Find TP53's module
if CONFIG.target_gene in modules["Gene"].values:
    tp53_module = modules[modules["Gene"] == CONFIG.target_gene]["Module"].values[0]
    print(f"\n*** TP53 is in module: {tp53_module} ***")
    tp53_module_genes = modules[modules["Module"] == tp53_module]["Gene"].tolist()
    print(f"Genes in TP53's module: {len(tp53_module_genes)}")
else:
    tp53_module = None
    print("TP53 not found in module assignments")

# Show module distribution
print("\nModule distribution:")
modules["Module"].value_counts()

[INFO] Loaded 2001 genes with module assignments
[INFO] Modules: ['lightgrey', 'darkgrey', 'silver', 'black', 'gainsboro', 'whitesmoke']
[INFO] Hub genes: 60



*** TP53 is in module: darkgrey ***
Genes in TP53's module: 625

Module distribution:


Module
darkgrey      625
lightgrey     600
black         230
gainsboro     224
whitesmoke    162
silver        160
Name: count, dtype: int64

## 4. Define Pathway Gene Sets
Create gene sets for key biological pathways related to TP53 function.

In [4]:
# Define curated pathway gene sets (based on KEGG/GO annotations)
PATHWAY_GENE_SETS = {
    "Apoptosis": {
        "TP53",
        "BAX",
        "PUMA",
        "NOXA",
        "BID",
        "APAF1",
        "CASP9",
        "CASP3",
        "BCL2",
        "BCL2L1",
        "MCL1",
        "BIRC5",
        "XIAP",
        "CYCS",
        "DIABLO",
        "ENDOG",
        "AIF",
        "HTRA2",
    },
    "Cell_Cycle": {
        "CDKN1A",
        "CDKN2A",
        "RB1",
        "E2F1",
        "E2F2",
        "CCND1",
        "CCNE1",
        "CDK2",
        "CDK4",
        "CDK6",
        "CCNB1",
        "CDC25A",
        "CDC25C",
        "PLK1",
        "AURKA",
        "AURKB",
        "BUB1",
        "MAD2L1",
        "CHEK1",
        "CHEK2",
    },
    "DNA_Damage_Response": {
        "ATM",
        "ATR",
        "BRCA1",
        "BRCA2",
        "RAD51",
        "XRCC1",
        "PARP1",
        "PARP2",
        "MLH1",
        "MSH2",
        "MSH6",
        "OGG1",
        "XPA",
        "XPC",
        "ERCC1",
        "ERCC2",
        "POLE",
        "POLD1",
        "FEN1",
        "LIG1",
        "LIG3",
    },
    "p53_Signaling": {
        "TP53",
        "MDM2",
        "CDKN1A",
        "BAX",
        "PUMA",
        "NOXA",
        "GADD45A",
        "SFN",
        "SERPINE1",
        "THBS1",
        "TSC2",
        "SESN1",
        "SESN2",
        "TIGAR",
        "SCO2",
    },
    "Metabolism": {
        "HIF1A",
        "LDHA",
        "PKM",
        "GLUT1",
        "HK2",
        "PFKFB3",
        "SCO2",
        "TIGAR",
        "GLS2",
        "PTEN",
        "AKT1",
        "MTOR",
        "AMPK",
        "PGC1A",
        "SIRT1",
        "SIRT3",
        "UCP2",
        "CPT1A",
        "ACACA",
        "FASN",
    },
}

# Show pathway sizes
pathway_summary = {name: len(genes) for name, genes in PATHWAY_GENE_SETS.items()}
pd.DataFrame([pathway_summary]).T.rename(columns={0: "Gene_Count"})

,Gene_Count
Apoptosis,18
Cell_Cycle,20
DNA_Damage_Response,21
p53_Signaling,15
Metabolism,20


## 4a. Online Enrichment via Enrichr (KEGG/GO/Reactome)
Use gseapy to query Enrichr - a web-based tool with access to 100+ gene set libraries including:
- **KEGG_2021_Human** - Metabolic and signaling pathways
- **GO_Biological_Process_2021** - Gene Ontology biological processes
- **Reactome_2022** - Curated pathway database
- **WikiPathways_2019_Human** - Community-curated pathways

In [5]:
def clean_gene_list(genes: List[str]) -> List[str]:
    """Clean gene names - handle multi-gene probe annotations like 'GENE1 /// GENE2'"""
    cleaned = []
    for g in genes:
        if " /// " in str(g):
            # Take first gene in multi-gene annotation
            cleaned.append(g.split(" /// ")[0])
        else:
            cleaned.append(str(g))
    return cleaned


def run_enrichr_analysis(
    gene_list: List[str],
    gene_sets: List[str] = None,
    organism: str = "Human",
    outdir: str = None,
) -> pd.DataFrame:
    """
    Run enrichment analysis using Enrichr (online database).
    """
    if not GSEAPY_AVAILABLE:
        logging.warning("gseapy not available")
        return pd.DataFrame()

    if gene_sets is None:
        gene_sets = ["KEGG_2021_Human", "GO_Biological_Process_2023", "Reactome_2022"]

    # Clean gene names
    gene_list = clean_gene_list(gene_list)

    logging.info(
        f"Running Enrichr with {len(gene_list)} genes against {len(gene_sets)} libraries"
    )

    all_results = []

    for lib in gene_sets:
        try:
            enr = gp.enrichr(
                gene_list=gene_list,
                gene_sets=lib,
                organism=organism,
                outdir=outdir,
                cutoff=0.5,
            )

            if len(enr.results) > 0:
                enr.results["Gene_set"] = lib
                all_results.append(enr.results)

        except Exception as e:
            logging.debug(f"Enrichr query for {lib} failed: {e}")
            continue

    if all_results:
        return pd.concat(all_results, ignore_index=True)
    return pd.DataFrame()


def run_module_enrichment_enrichr(
    modules_df: pd.DataFrame, gene_sets: List[str] = None, min_genes: int = 5
) -> Dict[str, pd.DataFrame]:
    """
    Run Enrichr enrichment for each module.
    """
    if not GSEAPY_AVAILABLE:
        return {}

    all_results = {}

    for module_name in sorted(modules_df["Module"].unique()):
        if module_name == "grey":  # Skip unassigned
            continue

        module_genes = modules_df[modules_df["Module"] == module_name]["Gene"].tolist()

        if len(module_genes) < min_genes:
            logging.info(
                f"Module {module_name}: skipping ({len(module_genes)} genes < {min_genes})"
            )
            continue

        logging.info(
            f"Module {module_name}: querying Enrichr with {len(module_genes)} genes"
        )

        results = run_enrichr_analysis(
            gene_list=module_genes,
            gene_sets=gene_sets,
            outdir=None,  # Don't save individual files
        )

        if len(results) > 0:
            results["Module"] = module_name
            all_results[module_name] = results

    return all_results


# Run online enrichment if gseapy is available
if GSEAPY_AVAILABLE:
    print("=" * 60)
    print("Running Enrichr Enrichment Analysis")
    print("=" * 60)
    print("Libraries: KEGG_2021_Human, GO_Biological_Process_2023, Reactome_2022\n")

    enrichr_results = run_module_enrichment_enrichr(
        modules,
        gene_sets=["KEGG_2021_Human", "GO_Biological_Process_2023", "Reactome_2022"],
        min_genes=CONFIG.min_genes_for_enrichment,
    )

    # Combine all module results
    if enrichr_results:
        enrichr_combined = pd.concat(enrichr_results.values(), ignore_index=True)
        print(f"\nTotal Enrichr results: {len(enrichr_combined)}")

        # Show top results per module
        print("\n" + "=" * 60)
        print("Top Enriched Terms per Module")
        print("=" * 60)

        for module_name, results in enrichr_results.items():
            sig = results[results["Adjusted P-value"] < 0.05]
            if len(sig) > 0:
                top = sig.nsmallest(5, "Adjusted P-value")[
                    ["Term", "Adjusted P-value", "Gene_set"]
                ]
                tp53_flag = (
                    " *** TP53's MODULE ***" if module_name == tp53_module else ""
                )
                print(f"\n{module_name}{tp53_flag}:")
                for _, row in top.iterrows():
                    term = (
                        row["Term"][:50] + "..."
                        if len(row["Term"]) > 50
                        else row["Term"]
                    )
                    print(f"  {term}: p={row['Adjusted P-value']:.2e}")
    else:
        enrichr_combined = pd.DataFrame()
        print("No Enrichr results obtained")
else:
    enrichr_combined = pd.DataFrame()
    print("Skipping online enrichment - gseapy not available")

[INFO] Module black: querying Enrichr with 230 genes
[INFO] Running Enrichr with 230 genes against 3 libraries


Running Enrichr Enrichment Analysis
Libraries: KEGG_2021_Human, GO_Biological_Process_2023, Reactome_2022



[INFO] Module darkgrey: querying Enrichr with 625 genes
[INFO] Running Enrichr with 625 genes against 3 libraries
[INFO] Module gainsboro: querying Enrichr with 224 genes
[INFO] Running Enrichr with 224 genes against 3 libraries
[INFO] Module lightgrey: querying Enrichr with 600 genes
[INFO] Running Enrichr with 600 genes against 3 libraries
[INFO] Module silver: querying Enrichr with 160 genes
[INFO] Running Enrichr with 160 genes against 3 libraries
[INFO] Module whitesmoke: querying Enrichr with 162 genes
[INFO] Running Enrichr with 162 genes against 3 libraries



Total Enrichr results: 14035

Top Enriched Terms per Module

black:
  Mitotic Sister Chromatid Segregation (GO:0000070): p=6.25e-09
  B Cell Receptor Signaling Pathway (GO:0050853): p=1.28e-08
  Antigen Receptor-Mediated Signaling Pathway (GO:00...: p=2.77e-08
  Microtubule Cytoskeleton Organization Involved In ...: p=7.07e-08
  Mitotic Cytokinesis (GO:0000281): p=7.07e-08

darkgrey *** TP53's MODULE ***:
  Surfactant Metabolism R-HSA-5683826: p=7.71e-06
  Diseases Associated With Surfactant Metabolism R-H...: p=2.89e-05
  Defective GALNT12 Causes CRCS1 R-HSA-5083636: p=3.35e-03
  Defective CSF2RA Causes SMDP4 R-HSA-5688890: p=6.39e-03
  Defective GALNT3 Causes HFTC R-HSA-5083625: p=2.97e-02

gainsboro:
  Viral protein interaction with cytokine and cytoki...: p=1.46e-08
  Immune System R-HSA-168256: p=1.49e-08
  Chemokine-Mediated Signaling Pathway (GO:0070098): p=2.74e-08
  Cellular Response To Chemokine (GO:1990869): p=2.74e-08
  Chemokine Receptors Bind Chemokines R-HSA-380108: p=1

## 5. Perform Enrichment Analysis
Use Fisher's exact test to assess pathway enrichment in each module.

In [6]:
def fisher_enrichment(
    module_genes: Set[str], pathway_genes: Set[str], background_genes: Set[str]
) -> Dict:
    """
    Perform Fisher's exact test for pathway enrichment.

    Returns dict with:
    - overlap: genes in both module and pathway
    - odds_ratio: enrichment score
    - pvalue: significance
    """
    # Ensure all sets are within background
    module_genes = module_genes & background_genes
    pathway_genes = pathway_genes & background_genes

    # Contingency table
    a = len(module_genes & pathway_genes)  # In module AND in pathway
    b = len(module_genes - pathway_genes)  # In module but NOT in pathway
    c = len(pathway_genes - module_genes)  # NOT in module but in pathway
    d = len(background_genes - module_genes - pathway_genes)  # Neither

    # Fisher's exact test
    contingency = [[a, b], [c, d]]
    odds_ratio, pvalue = stats.fisher_exact(contingency, alternative="greater")

    return {
        "overlap_count": a,
        "overlap_genes": module_genes & pathway_genes,
        "module_size": len(module_genes),
        "pathway_size": len(pathway_genes),
        "odds_ratio": odds_ratio,
        "pvalue": pvalue,
    }


def run_enrichment_analysis(
    modules_df: pd.DataFrame, pathway_sets: Dict[str, Set[str]], min_genes: int = 5
) -> pd.DataFrame:
    """
    Run enrichment analysis for all modules against all pathways.
    """
    background = set(modules_df["Gene"])
    results = []

    for module_id in sorted(modules_df["Module"].unique()):
        if module_id == 0:  # Skip unassigned
            continue

        module_genes = set(modules_df[modules_df["Module"] == module_id]["Gene"])

        if len(module_genes) < min_genes:
            continue

        for pathway_name, pathway_genes in pathway_sets.items():
            enrichment = fisher_enrichment(module_genes, pathway_genes, background)

            results.append(
                {
                    "Module": module_id,
                    "Pathway": pathway_name,
                    "Overlap": enrichment["overlap_count"],
                    "ModuleSize": enrichment["module_size"],
                    "PathwaySize": enrichment["pathway_size"],
                    "OddsRatio": enrichment["odds_ratio"],
                    "PValue": enrichment["pvalue"],
                    "OverlapGenes": ", ".join(sorted(enrichment["overlap_genes"])),
                }
            )

    results_df = pd.DataFrame(results)

    # Adjust p-values (Benjamini-Hochberg)
    if len(results_df) > 0:
        from scipy.stats import false_discovery_control

        try:
            results_df["AdjPValue"] = false_discovery_control(results_df["PValue"])
        except:
            # Fallback: simple Bonferroni correction
            results_df["AdjPValue"] = results_df["PValue"] * len(results_df)
            results_df["AdjPValue"] = results_df["AdjPValue"].clip(upper=1.0)

    return results_df.sort_values("PValue")


enrichment_results = run_enrichment_analysis(
    modules, PATHWAY_GENE_SETS, min_genes=CONFIG.min_genes_for_enrichment
)

print(f"Enrichment analysis complete: {len(enrichment_results)} tests performed")
enrichment_results.head(15)

Enrichment analysis complete: 30 tests performed


,Module,Pathway,Overlap,ModuleSize,PathwaySize,OddsRatio,PValue,OverlapGenes,AdjPValue
14,gainsboro,Metabolism,1,224,1,inf,0.111944,HK2,1.0
0,black,Apoptosis,1,230,2,7.729258,0.216724,BIRC5,1.0
28,whitesmoke,p53_Signaling,1,162,3,5.704969,0.223848,SERPINE1,1.0
11,gainsboro,Cell_Cycle,1,224,3,3.979821,0.299773,CDKN2A,1.0
17,lightgrey,DNA_Damage_Response,1,600,1,inf,0.299850,MSH2,1.0
1,black,Cell_Cycle,1,230,3,3.862445,0.306846,CCNB1,1.0
5,darkgrey,Apoptosis,1,625,2,2.203526,0.527236,TP53,1.0
18,lightgrey,p53_Signaling,1,600,3,1.167780,0.657000,SFN,1.0
6,darkgrey,Cell_Cycle,1,625,3,1.100962,0.675049,CCND1,1.0
8,darkgrey,p53_Signaling,1,625,3,1.100962,0.675049,TP53,1.0


In [7]:
# Show significant enrichments
significant = enrichment_results[enrichment_results["PValue"] < CONFIG.pvalue_threshold]
print(f"\nSignificant enrichments (p < {CONFIG.pvalue_threshold}): {len(significant)}")
significant


Significant enrichments (p < 0.05): 0


,Module,Pathway,Overlap,ModuleSize,PathwaySize,OddsRatio,PValue,OverlapGenes,AdjPValue


## 6. Module Characterization
Summarize each module's biological function based on enrichment results.

In [8]:
def characterize_modules(
    modules_df: pd.DataFrame,
    enrichr_results: Dict[str, pd.DataFrame],
    hubs_df: pd.DataFrame,
    pvalue_threshold: float = 0.05,
) -> pd.DataFrame:
    """
    Create a summary characterization of each module using Enrichr results.
    """
    summaries = []

    for module_name in sorted(modules_df["Module"].unique()):
        if module_name == "grey":
            continue

        # Module genes
        module_genes = modules_df[modules_df["Module"] == module_name]["Gene"].tolist()

        # Hub genes in this module
        if "IsHub" in hubs_df.columns:
            module_hubs = hubs_df[
                (hubs_df["Gene"].isin(module_genes)) & (hubs_df["IsHub"])
            ]["Gene"].tolist()
        else:
            module_hubs = hubs_df[hubs_df["Gene"].isin(module_genes)]["Gene"].tolist()

        # Get enrichment results for this module
        if module_name in enrichr_results:
            mod_enr = enrichr_results[module_name]
            sig = mod_enr[mod_enr["Adjusted P-value"] < pvalue_threshold]

            if len(sig) > 0:
                top_term = sig.iloc[0]["Term"]
                top_pvalue = sig.iloc[0]["Adjusted P-value"]
                sig_pathways = sig["Term"].head(5).tolist()
            else:
                top_term = "None significant"
                top_pvalue = 1.0
                sig_pathways = []
        else:
            top_term = "Not tested"
            top_pvalue = 1.0
            sig_pathways = []

        summaries.append(
            {
                "Module": module_name,
                "Size": len(module_genes),
                "NumHubs": len(module_hubs),
                "HubGenes": ", ".join(module_hubs[:5]),
                "TopPathway": top_term[:50] if len(top_term) > 50 else top_term,
                "TopPValue": top_pvalue,
                "SignificantPathways": len(sig_pathways),
            }
        )

    return pd.DataFrame(summaries)


# Create module summary from Enrichr results
module_summary = characterize_modules(
    modules, enrichr_results if GSEAPY_AVAILABLE else {}, hubs, CONFIG.pvalue_threshold
)

print("\nModule Characterization Summary:")
module_summary


Module Characterization Summary:


,Module,Size,NumHubs,HubGenes,TopPathway,TopPValue,SignificantPathways
0,black,230,10,IGH /// IGHA1 /// IGHA2 /// IGHG1 /// IGHG2 //...,Primary immunodeficiency,2.947977e-04,5
1,darkgrey,625,10,"SFTA2, SFTA3, NAPSA, C16orf89, FOLR1",Surfactant Metabolism R-HSA-5683826,7.714232e-06,5
2,gainsboro,224,10,"CDC20, TPX2, FOXM1, RRM2, KIF18B",Viral protein interaction with cytokine and cy...,1.455268e-08,5
3,lightgrey,600,10,"KRT5, KRT6A /// KRT6B /// KRT6C, KRT6B, DSG3, ...",IL-17 signaling pathway,2.696533e-05,5
4,silver,160,10,"RPS4Y1, XIST, KDM5D, EIF1AY, FGA",Plasminogen Activation (GO:0031639),1.563076e-02,5
5,whitesmoke,162,10,"CTSK, COL5A1, MXRA5, C1QB, GLT8D2",Staphylococcus aureus infection,4.650390e-08,5


## 7. TP53-Specific Analysis
Analyze TP53's position in the network and its module associations.

In [9]:
def analyze_tp53(
    modules_df: pd.DataFrame, hubs_df: pd.DataFrame, target_gene: str = "TP53"
) -> Dict:
    """
    Analyze TP53's role in the network.
    """
    # Find TP53 in modules
    tp53_module_data = modules_df[modules_df["Gene"] == target_gene]

    if len(tp53_module_data) == 0:
        return {"error": f"{target_gene} not found in dataset"}

    tp53_module = tp53_module_data.iloc[0]["Module"]

    # Get co-module genes
    co_module_genes = modules_df[modules_df["Module"] == tp53_module]["Gene"].tolist()

    # Get hub info
    if "IsHub" in hubs_df.columns:
        tp53_hub_data = hubs_df[hubs_df["Gene"] == target_gene]
        is_hub = tp53_hub_data["IsHub"].values[0] if len(tp53_hub_data) > 0 else False
        connectivity = (
            tp53_hub_data["IntramodularConnectivity"].values[0]
            if len(tp53_hub_data) > 0
            else 0
        )

        co_module_hubs = hubs_df[
            (hubs_df["Gene"].isin(co_module_genes)) & (hubs_df["IsHub"])
        ]["Gene"].tolist()
    else:
        is_hub = target_gene in hubs_df["Gene"].values
        connectivity = 0
        co_module_hubs = hubs_df[hubs_df["Gene"].isin(co_module_genes)]["Gene"].tolist()

    return {
        "gene": target_gene,
        "module": tp53_module,
        "is_hub": is_hub,
        "intramodular_connectivity": connectivity,
        "module_size": len(co_module_genes),
        "module_hubs": co_module_hubs[:10],
        "module_genes_sample": co_module_genes[:15],
    }


tp53_analysis = analyze_tp53(modules, hubs, CONFIG.target_gene)

print("\n" + "=" * 60)
print(f"{CONFIG.target_gene} Network Analysis")
print("=" * 60)
for key, value in tp53_analysis.items():
    if isinstance(value, list):
        print(f"  {key}: {value[:5]}..." if len(value) > 5 else f"  {key}: {value}")
    else:
        print(f"  {key}: {value}")


TP53 Network Analysis
  gene: TP53
  module: darkgrey
  is_hub: False
  intramodular_connectivity: 0.0
  module_size: 625
  module_hubs: ['SFTA2', 'SFTA3', 'NAPSA', 'C16orf89', 'FOLR1']...
  module_genes_sample: ['SCGB1A1', 'SPINK1', 'SCGB3A2', 'SFTA2', 'SFTA3']...


## 8. Biological Interpretation
Generate a structured interpretation of the results.

In [10]:
def generate_interpretation(
    module_summary: pd.DataFrame,
    enrichr_results: Dict[str, pd.DataFrame],
    tp53_analysis: Dict,
) -> Dict:
    """
    Generate structured biological interpretation.
    """
    interpretation = {
        "network_overview": {
            "total_modules": len(module_summary) if len(module_summary) > 0 else 0,
            "total_hub_genes": module_summary["NumHubs"].sum()
            if len(module_summary) > 0
            else 0,
            "modules_with_significant_enrichment": len(
                module_summary[module_summary["TopPValue"] < 0.05]
            )
            if len(module_summary) > 0
            else 0,
        },
        "tp53_findings": {
            "module": tp53_analysis.get("module", "Unknown"),
            "is_hub": tp53_analysis.get("is_hub", False),
            "module_size": tp53_analysis.get("module_size", 0),
            "co_expressed_genes": tp53_analysis.get("module_genes_sample", [])[:10],
        },
        "key_modules": [],
    }

    # Characterize key modules from Enrichr results
    if len(module_summary) > 0:
        for _, row in module_summary.iterrows():
            if row["TopPValue"] < 0.05:
                interpretation["key_modules"].append(
                    {
                        "module_name": row["Module"],
                        "size": row["Size"],
                        "primary_function": row["TopPathway"],
                        "hub_genes": row["HubGenes"],
                        "is_tp53_module": row["Module"] == tp53_analysis.get("module"),
                    }
                )

    return interpretation


if len(module_summary) > 0:
    interpretation = generate_interpretation(
        module_summary, enrichr_results, tp53_analysis
    )
else:
    interpretation = {
        "network_overview": {"note": "No module summary available"},
        "tp53_findings": tp53_analysis,
        "key_modules": [],
    }

print("\n" + "=" * 60)
print("BIOLOGICAL INTERPRETATION SUMMARY")
print("=" * 60)

print(f"\nNetwork Overview:")
for key, value in interpretation["network_overview"].items():
    print(f"  - {key.replace('_', ' ').title()}: {value}")

print(f"\nTP53 Findings:")
for key, value in interpretation["tp53_findings"].items():
    if isinstance(value, list):
        print(
            f"  - {key.replace('_', ' ').title()}: {value[:5]}..."
            if len(value) > 5
            else f"  - {key.replace('_', ' ').title()}: {value}"
        )
    else:
        print(f"  - {key.replace('_', ' ').title()}: {value}")

print(f"\nKey Modules with Significant Enrichment:")
for mod in interpretation["key_modules"]:
    tp53_flag = " *** TP53's MODULE ***" if mod.get("is_tp53_module") else ""
    print(f"  {mod['module_name']}{tp53_flag}: {mod['primary_function']}")
    print(f"    Size: {mod['size']}, Hub genes: {mod['hub_genes']}")


BIOLOGICAL INTERPRETATION SUMMARY

Network Overview:
  - Total Modules: 6
  - Total Hub Genes: 60
  - Modules With Significant Enrichment: 6

TP53 Findings:
  - Module: darkgrey
  - Is Hub: False
  - Module Size: 625
  - Co Expressed Genes: ['SCGB1A1', 'SPINK1', 'SCGB3A2', 'SFTA2', 'SFTA3']...

Key Modules with Significant Enrichment:
  black: Primary immunodeficiency
    Size: 230, Hub genes: IGH /// IGHA1 /// IGHA2 /// IGHG1 /// IGHG2 /// IGHG3 /// IGHM /// IGHV4-31 /// LOC102725526, IGHA1 /// IGHD /// IGHG1 /// IGHG3 /// IGHM /// IGHV4-31, MZB1, IGLL5, IGH /// IGHA1 /// IGHD /// IGHG1 /// IGHG3 /// IGHM /// IGHV3-23 /// IGHV4-31
  darkgrey *** TP53's MODULE ***: Surfactant Metabolism R-HSA-5683826
    Size: 625, Hub genes: SFTA2, SFTA3, NAPSA, C16orf89, FOLR1
  gainsboro: Viral protein interaction with cytokine and cytoki
    Size: 224, Hub genes: CDC20, TPX2, FOXM1, RRM2, KIF18B
  lightgrey: IL-17 signaling pathway
    Size: 600, Hub genes: KRT5, KRT6A /// KRT6B /// KRT6C, KRT6B, 

## 9. Export Results
Save enrichment results and interpretations.

In [11]:
EXPORT_DIR = CONFIG.export_dir
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

# Save Enrichr results (online KEGG/GO/Reactome)
if GSEAPY_AVAILABLE and len(enrichr_combined) > 0:
    enrichr_combined.to_csv(
        EXPORT_DIR / "task5_enrichr_kegg_go_results.csv", index=False
    )
    print(
        f"[OK] Enrichr (KEGG/GO/Reactome) results saved to: {EXPORT_DIR / 'task5_enrichr_kegg_go_results.csv'}"
    )

    # Save significant Enrichr results only
    enrichr_significant = enrichr_combined[enrichr_combined["Adjusted P-value"] < 0.05]
    enrichr_significant.to_csv(
        EXPORT_DIR / "task5_enrichr_significant.csv", index=False
    )
    print(f"[OK] Significant Enrichr results ({len(enrichr_significant)} terms) saved")

    # Save per-module enrichment
    for mod_name, mod_results in enrichr_results.items():
        mod_results.to_csv(EXPORT_DIR / f"task5_enrichment_{mod_name}.csv", index=False)
    print(f"[OK] Per-module enrichment files saved")

# Save module summary
if len(module_summary) > 0:
    module_summary.to_csv(EXPORT_DIR / "task5_module_summary.csv", index=False)
    print(f"[OK] Module summary saved to: {EXPORT_DIR / 'task5_module_summary.csv'}")

# Save interpretation as JSON
import json


def convert_to_serializable(obj):
    """Convert numpy types to Python native types for JSON serialization."""
    if isinstance(obj, dict):
        return {k: convert_to_serializable(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [convert_to_serializable(item) for item in obj]
    elif isinstance(obj, (np.integer, np.int64, np.int32)):
        return int(obj)
    elif isinstance(obj, (np.floating, np.float64, np.float32)):
        return float(obj)
    elif isinstance(obj, np.ndarray):
        return obj.tolist()
    elif isinstance(obj, (np.bool_, bool)):
        return bool(obj)
    return obj


with open(EXPORT_DIR / "task5_interpretation.json", "w") as f:
    json.dump(convert_to_serializable(interpretation), f, indent=2)
print(f"[OK] Interpretation saved to: {EXPORT_DIR / 'task5_interpretation.json'}")

# Save TP53 analysis
pd.DataFrame(
    [{k: str(v) if isinstance(v, list) else v for k, v in tp53_analysis.items()}]
).to_csv(EXPORT_DIR / "task5_tp53_analysis.csv", index=False)
print(f"[OK] TP53 analysis saved to: {EXPORT_DIR / 'task5_tp53_analysis.csv'}")

print(f"\n{'=' * 60}")
print("TASK 5 COMPLETE - Enrichment Analysis")
print(f"{'=' * 60}")
print(f"Modules analyzed: {len(module_summary)}")
print(f"TP53 module: {tp53_analysis.get('module', 'N/A')}")
if GSEAPY_AVAILABLE and len(enrichr_combined) > 0:
    print(f"Total enrichment terms: {len(enrichr_combined)}")
    print(f"Significant terms (p<0.05): {len(enrichr_significant)}")

[OK] Enrichr (KEGG/GO/Reactome) results saved to: artifacts/task5_enrichr_kegg_go_results.csv
[OK] Significant Enrichr results (617 terms) saved
[OK] Per-module enrichment files saved
[OK] Module summary saved to: artifacts/task5_module_summary.csv
[OK] Interpretation saved to: artifacts/task5_interpretation.json
[OK] TP53 analysis saved to: artifacts/task5_tp53_analysis.csv

TASK 5 COMPLETE - Enrichment Analysis
Modules analyzed: 6
TP53 module: darkgrey
Total enrichment terms: 14035
Significant terms (p<0.05): 617
